In [5]:
import os, sys, torch
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath('../../modules'))
sys.path.append(os.path.abspath('../../modules/imagenet'))
import train as tt
import ortho as uno
import surgery as ts
import ortho_s as tos
# import vae_ascent as va
# import vae_ad as vad
import classifier as cl
import dit

import batch as bt
import utility as ut

device = ut.get_device()


# load custom modules
import classifier as cl
import batch as bt
import utility as ut
import datapipe


# set general parameters
device = ut.get_device()
imagenet_folder = "../../data/ImageNet"
experiment_folder = "../../data/ImageNet/ImageNet-Experiments"
num_fid_samples = 25000
fid_batch_size = 256
num_experiments = 10
model_path = f"{imagenet_folder}/DiT-XL-2"
imagenet_json_path = "../../data/ImageNet/imagenet_class_index.json"
data_path = f"{imagenet_folder}/2012"

In [6]:
# set more experiment parameters
forget_class = 207
exchange_classes = [208]
num_fid_samples = 15000
fid_batch_size = 64
num_experiments = 10

params = {
    "model_path": model_path,
    "num_steps": 100,
    "batch_size": 10,
    "log_interval": 1,
    "collect_interval": "epoch",
    "save_steps": None,
    "exchange_classes": exchange_classes,
    "forget_class": forget_class,
    "data_path": data_path,
    "imagenet_json_path": imagenet_json_path,
    "freeze_K": 4,
    "unfreeze_last": True,
    "keep_all": True,
    "n_samples": 50,
    "device": device,
    "diffusion_steps": 15,
    "learning_rate": 1e-4,
}

gen_kwargs = {
    "guidance_scale": 8.0,
    "n_steps": 15,
}

gen_kwargs_summary = {
    "guidance_scale": 2.5,
    "n_steps": 10,

}

threshold = 0.18

**Batch Experiments: S**

In [7]:
train_kwargs = params | {"orthogonality_weight": 2e-2} | gen_kwargs
train_kwargs["folder"] = f"{experiment_folder}/dit-s"
be = bt.BatchExperiment(ts.train, train_kwargs, num_experiments, **gen_kwargs_summary)
# be.run()
# torch.cuda.empty_cache()
# be.fid(num_fid_samples, device, fid_batch_size)
be.summarize_wo_fid(threshold)#num_fid_samples=num_fid_samples, batch_size=256)

({'Step': np.float64(10.7),
  'Total Loss': np.float64(0.19555993080139156),
  'Time': np.float64(7.622259095978737),
  '0 Fraction': np.float64(0.892),
  '1 Fraction': np.float64(0.10800000000000001),
  'FID': np.float64(12.254233407449902),
  'Time/Step': np.float64(0.7123606631755829)},
 {'Step': np.float64(3.1),
  'Total Loss': np.float64(0.07687660637789645),
  'Time': np.float64(2.21058862827063),
  '0 Fraction': np.float64(0.04578209256903839),
  '1 Fraction': np.float64(0.04578209256903839),
  'FID': np.float64(0.7667806459332257),
  'Time/Step': np.float64(0.008991646078769315)})

**Batch Experiments: O**

In [8]:
train_kwargs = params | {"orthogonality_weight": 2e-2} | gen_kwargs
train_kwargs["folder"] = f"{experiment_folder}/dit-o"
be = bt.BatchExperiment(uno.train, train_kwargs, num_experiments, **gen_kwargs_summary)
# be.run()
# torch.cuda.empty_cache()
# be.fid(num_fid_samples, device, fid_batch_size)
be.summarize_wo_fid(threshold)#num_fid_samples=num_fid_samples, batch_size=256)

({'Step': np.float64(3.5),
  'Total Loss': np.float64(0.3206746742129326),
  'Time': np.float64(6.360512320399284),
  '0 Fraction': np.float64(0.97),
  '1 Fraction': np.float64(0.030000000000000006),
  'Orthogonality Loss': np.float64(0.15566349886357783),
  'FID': np.float64(11.845353195168226),
  'Time/Step': np.float64(1.8172892343997955)},
 {'Step': np.float64(0.5),
  'Total Loss': np.float64(0.1340944993508768),
  'Time': np.float64(0.9087892054881153),
  '0 Fraction': np.float64(0.0412310562561766),
  '1 Fraction': np.float64(0.0412310562561766),
  'Orthogonality Loss': np.float64(0.14098400911114206),
  'FID': np.float64(0.8135952687715895),
  'Time/Step': np.float64(0.004585015990123001)})

**Batch Experiments: OS**

In [29]:
train_kwargs = params | {"orthogonality_weight": 2e-2} | gen_kwargs
train_kwargs["folder"] = f"{experiment_folder}/dit-os"
be = bt.BatchExperiment(tos.train, train_kwargs, num_experiments, **gen_kwargs_summary)
# be.run()
# torch.cuda.empty_cache()
# be.fid(num_fid_samples, device, fid_batch_size)
be.summarize_wo_fid(threshold)#num_fid_samples=num_fid_samples, batch_size=256)

({'Step': np.float64(235.5),
  'Total Loss': np.float64(0.19765380844473837),
  'Time': np.float64(308.8159984441996),
  '0 Fraction': np.float64(0.8540000000000001),
  '1 Fraction': np.float64(0.14600000000000002),
  'FID': np.float64(13.493452656847884),
  'Time/Step': np.float64(1.3113205878734588)},
 {'Step': np.float64(114.6178432880326),
  'Total Loss': np.float64(0.09952320706629499),
  'Time': np.float64(150.33147487263187),
  '0 Fraction': np.float64(0.0200997512422418),
  '1 Fraction': np.float64(0.02009975124224178),
  'FID': np.float64(0.21722892045961362),
  'Time/Step': np.float64(0.011606358620614259)})

In [30]:
suffix =  ["s", "o", "os"]#, "hat", "shat", "ohat", "ohatshat"]

ut.summary_csv(\
[f"{experiment_folder}/dit-{s}/summary.json" for s in suffix],\
[f"{experiment_folder}/dit-{s}/summary_std.json" for s in suffix], suffix,\
 f"{experiment_folder}/summary.csv")